# mBERT Optuna — Armenian Participle Punctuation (Colab T4)

**Task:** 4-class token classification (`O`, `COMMA_AFTER`, `BUTH_AFTER`, `REMOVE_COMMA`)  
**Model:** `bert-base-multilingual-cased` (110M params, 12 layers)  
**Backbone upgrade from:** `aking11/hyebert` (66M, 6 layers, ceiling F1=0.41)

**Session workflow:**
1. Steps 1–2 re-run quickly on every session restart (~30s)
2. Step 3 automatically resumes Optuna from the last completed trial
3. Steps 4–5 only run once, after all trials finish

**Expected timing:** ~1.8 h/trial on T4 bs=16 → ~6–7 trials per 12h session → 15 trials in ~2.5 sessions.

## Step 1: Mount Drive & Install Dependencies

In [1]:
!pip install optuna transformers datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch, os
assert torch.cuda.is_available(), "No GPU detected — switch to a T4 runtime!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi -L

GPU: Tesla T4
VRAM: 15.6 GB
GPU 0: Tesla T4 (UUID: GPU-47cb9507-431c-0e6e-9957-2d578e94a10d)


In [4]:
# Suppress noisy transformers logging
import logging, warnings, os
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Logging suppressed.")

Logging suppressed.


## Step 2: Load Data (runs fast, ~30s)

Expects worker JSONL files at `drive/MyDrive/armenian_punct/results_clean/worker_*_clean.jsonl`.  
Upload the 7 worker files to Google Drive before running.

In [20]:
# ── Configuration (must match v3 exactly) ──────────────────────
import re, difflib, json, math, random, time
from pathlib import Path
from collections import Counter
from datetime import datetime
from sklearn.metrics import f1_score, classification_report
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification

LABEL_MAP  = {"O": 0, "COMMA_AFTER": 1, "BUTH_AFTER": 2, "REMOVE_COMMA": 3}
LABEL_LIST = ["O", "COMMA_AFTER", "BUTH_AFTER", "REMOVE_COMMA"]
NUM_LABELS = 4

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128
NEGATIVE_KEEP_RATIO = 0.15
SEED = 42
N_TRIALS = 15

BASE_DIR   = Path("drive/MyDrive/Colab Notebooks/Capstone_Data/armenian_punct")
DATA_DIR   = BASE_DIR / "results_clean"
STUDY_DB   = f"sqlite:///{BASE_DIR / 'mbert_optuna.db'}"
TRIAL_LOG  = str(BASE_DIR / "mbert_trial_log.json")
CONFIG_OUT = str(BASE_DIR / "mbert_best_config.json")

TRAIN_WORKERS = [f"worker_{i}_clean.jsonl" for i in [0, 1, 2, 3, 6]]
VAL_WORKERS   = [f"worker_{i}_clean.jsonl" for i in [4, 5]]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda")
print(f"Config OK — model={MODEL_NAME}, device={DEVICE}")

Config OK — model=bert-base-multilingual-cased, device=cuda


In [6]:
# ── Buth-fixed tokenizer & difflib label pipeline ─────────────
ARMENIAN_PUNCT = '\u055b\u055c\u055d\u055e\u055f\u0589'

def tokenize_armenian(text):
    """Splits Armenian text into word and punctuation tokens.
    Buth-fixed: pre-separates Armenian punctuation marks that fall
    inside the Unicode Armenian block (U+0530-058F) before regex."""
    if not isinstance(text, str) or not text.strip():
        return []
    for ch in ARMENIAN_PUNCT:
        text = text.replace(ch, f' {ch} ')
    return re.findall(r'[\w\u0530-\u058F]+|[^\w\s]', text)


def generate_bilstm_labels(orig_tokens, corr_tokens):
    """Difflib-based label generation — exact BiLSTM v2 pipeline."""
    labels = [0] * len(orig_tokens)
    sm = difflib.SequenceMatcher(None, orig_tokens, corr_tokens)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            continue
        if tag in ('delete', 'replace'):
            for i in range(i1, i2):
                if orig_tokens[i] == ',':
                    labels[i] = 3  # REMOVE_COMMA
        if tag in ('insert', 'replace'):
            for j in range(j1, j2):
                tok = corr_tokens[j]
                if tok == ',' and i1 > 0:
                    labels[i1 - 1] = 1  # COMMA_AFTER
                elif tok == '\u055d' and i1 > 0:
                    labels[i1 - 1] = 2  # BUTH_AFTER
    return labels


def _is_punct(tok):
    """Check if a token is punctuation (including Armenian punct chars)."""
    if tok in set(ARMENIAN_PUNCT):
        return True
    return not bool(re.search(r'[\w\u0530-\u058F]', tok))


def extract_word_labels(orig_tokens, label_ids):
    """Extract word-only tokens and labels, skipping punctuation.
    REMOVE_COMMA on a punct token is moved to the preceding word."""
    words, labels = [], []
    for i, (tok, lbl) in enumerate(zip(orig_tokens, label_ids)):
        if _is_punct(tok):
            # If this punct has REMOVE_COMMA, transfer to preceding word
            # BUT don't overwrite an existing insertion label (BUTH_AFTER/COMMA_AFTER)
            # — comma→buth replacements set both labels; the insertion wins.
            if lbl == 3 and labels and labels[-1] == 0:
                labels[-1] = 3
            continue
        words.append(tok)
        labels.append(lbl)
    return words, labels

print("Label pipeline ready.")

Label pipeline ready.


In [7]:
# ── Load worker JSONL files ─────────────────────────────────────
def load_workers(worker_list):
    records = []
    for wf in worker_list:
        path = DATA_DIR / wf
        assert path.exists(), f"Missing: {path}"
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                records.append(rec)
    return records

raw_train = load_workers(TRAIN_WORKERS)
raw_val   = load_workers(VAL_WORKERS)
print(f"Loaded: {len(raw_train)} train, {len(raw_val)} val records")

Loaded: 101047 train, 11026 val records


In [8]:
# ── Generate labels for each record ────────────────────────────
def process_record(rec):
    """Returns (words, labels) or None if both texts are empty."""
    orig = rec.get("original", "") or ""
    corr = rec.get("corrected", "") or rec.get("corrected_sentence", "") or ""
    if not orig.strip():
        return None
    if not corr.strip():
        corr = orig  # no correction → all O

    orig_tokens = tokenize_armenian(orig)
    corr_tokens = tokenize_armenian(corr)
    if not orig_tokens:
        return None

    raw_labels = generate_bilstm_labels(orig_tokens, corr_tokens)
    words, labels = extract_word_labels(orig_tokens, raw_labels)
    if not words:
        return None
    return {"words": words, "labels": labels}


def process_all(records):
    processed = []
    skipped = 0
    for rec in records:
        result = process_record(rec)
        if result is None:
            skipped += 1
            continue
        processed.append(result)
    return processed, skipped

train_data, skip_train = process_all(raw_train)
val_data, skip_val     = process_all(raw_val)
print(f"Processed: {len(train_data)} train ({skip_train} skipped), "
      f"{len(val_data)} val ({skip_val} skipped)")

Processed: 101047 train (0 skipped), 11026 val (0 skipped)


In [9]:
# ── Negative filtering (keep NEGATIVE_KEEP_RATIO of all-O sentences) ──
def filter_negatives(data, keep_ratio, seed=SEED):
    positives, negatives = [], []
    for rec in data:
        if any(l != 0 for l in rec["labels"]):
            positives.append(rec)
        else:
            negatives.append(rec)
    rng = random.Random(seed)
    kept_neg = rng.sample(negatives, int(len(negatives) * keep_ratio))
    combined = positives + kept_neg
    rng.shuffle(combined)
    return combined

train_filtered = filter_negatives(train_data, NEGATIVE_KEEP_RATIO)
print(f"After 15% neg filtering: {len(train_filtered)} train sentences")

After 15% neg filtering: 35517 train sentences


In [10]:
# ── Label distribution report ──────────────────────────────────
def report_distribution(data, split_name):
    counts = Counter()
    for rec in data:
        counts.update(rec["labels"])
    total = sum(counts.values())
    print(f"\n{split_name} label distribution ({total:,} tokens):")
    for i, name in enumerate(LABEL_LIST):
        c = counts.get(i, 0)
        print(f"  {name:15s}: {c:>10,} ({100*c/total:.2f}%)")
    return counts

train_counts = report_distribution(train_filtered, "TRAIN (filtered 15%)")
val_counts   = report_distribution(val_data, "VAL (unfiltered)")


TRAIN (filtered 15%) label distribution (787,281 tokens):
  O              :    759,031 (96.41%)
  COMMA_AFTER    :      9,126 (1.16%)
  BUTH_AFTER     :     18,297 (2.32%)
  REMOVE_COMMA   :        827 (0.11%)

VAL (unfiltered) label distribution (245,144 tokens):
  O              :    241,960 (98.70%)
  COMMA_AFTER    :      1,113 (0.45%)
  BUTH_AFTER     :      1,959 (0.80%)
  REMOVE_COMMA   :        112 (0.05%)


In [11]:
# ── Class weights: sqrt inverse frequency ──────────────────────
def compute_sqrt_inv_freq_weights(counts):
    total = sum(counts.values())
    weights = {}
    for i, name in enumerate(LABEL_LIST):
        c = counts.get(i, 1)
        weights[name] = math.sqrt(total / (NUM_LABELS * c))
    min_w = min(weights.values())
    normed = {k: v / min_w for k, v in weights.items()}
    wt = torch.tensor([normed[n] for n in LABEL_LIST], dtype=torch.float32)
    print("\nSqrt inverse frequency weights:")
    for i, name in enumerate(LABEL_LIST):
        print(f"  {name:15s}: {wt[i]:>8.2f}")
    return wt

CLASS_WEIGHTS = compute_sqrt_inv_freq_weights(train_counts).to(DEVICE)


Sqrt inverse frequency weights:
  O              :     1.00
  COMMA_AFTER    :     9.12
  BUTH_AFTER     :     6.44
  REMOVE_COMMA   :    30.30


In [12]:
# ── Tokenizer & Dataset ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PunctDataset(Dataset):
    """Token classification dataset with WordPiece subword alignment."""
    def __init__(self, data, tokenizer, max_length=MAX_LENGTH):
        self.samples = []
        for rec in data:
            words, labels = rec["words"], rec["labels"]
            encoding = tokenizer(
                words,
                is_split_into_words=True,
                max_length=max_length,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            word_ids = encoding.word_ids(batch_index=0)
            aligned_labels = []
            prev_word_id = None
            for wid in word_ids:
                if wid is None:
                    aligned_labels.append(-100)
                elif wid != prev_word_id:
                    aligned_labels.append(labels[wid] if wid < len(labels) else 0)
                else:
                    aligned_labels.append(-100)  # subword continuation
                prev_word_id = wid
            self.samples.append({
                "input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0),
                "labels": torch.tensor(aligned_labels, dtype=torch.long),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_dataset = PunctDataset(train_filtered, tokenizer)
val_dataset   = PunctDataset(val_data, tokenizer)
print(f"Datasets built: {len(train_dataset)} train, {len(val_dataset)} val samples")
print("Step 2 complete ✓")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Datasets built: 35517 train, 11026 val samples
Step 2 complete ✓


## Step 3: Run/Resume Optuna (THE LONG PART — run this, let it go)

Each trial trains for 3 epochs with patience=2.  
~1.8 hours per trial on T4 at bs=16.  
Progress is saved to Google Drive after every trial (SQLite + JSON backup).  
If the session disconnects, just re-run Steps 1–3 to resume.

In [ ]:
import optuna
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import f1_score

# ── Training function ──────────────────────────────────────────
def train_and_evaluate(config, train_ds, val_ds, class_weights):
    """Train mBERT for one Optuna trial. Returns best macro-F1."""
    lr          = config["lr"]
    bs          = config["batch_size"]
    warmup      = config["warmup_ratio"]
    freeze_n    = config["freeze_layers"]
    n_epochs    = 3
    patience    = 2

    # -- Model --
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).to(DEVICE)

    # Freeze bottom N encoder layers
    if freeze_n > 0:
        for i in range(freeze_n):
            for param in model.bert.encoder.layer[i].parameters():
                param.requires_grad = False

    # -- Optimizer (no weight decay on bias/LayerNorm) --
    no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}
    param_groups = [
        {"params": [p for n, p in model.named_parameters()
                     if p.requires_grad and not any(nd in n for nd in no_decay)],
         "weight_decay": 0.01},
        {"params": [p for n, p in model.named_parameters()
                     if p.requires_grad and any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    optimizer = AdamW(param_groups, lr=lr)

    # -- Warmup scheduler --
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=2, pin_memory=True)
    total_steps = len(train_loader) * n_epochs
    warmup_steps = int(total_steps * warmup)

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / max(1, warmup_steps)
        return max(0.0, float(total_steps - step) / max(1, total_steps - warmup_steps))

    scheduler = LambdaLR(optimizer, lr_lambda)

    # -- Loss --
    loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)

    # -- Val loader --
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False,
                            num_workers=2, pin_memory=True)

    best_f1 = 0.0
    patience_counter = 0

    for epoch in range(n_epochs):
        # ---- Train ----
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels    = batch["labels"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            logits  = outputs.logits  # (B, seq_len, num_labels)

            loss = loss_fn(logits.view(-1, NUM_LABELS), labels.view(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)

        # ---- Validate ----
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attn_mask = batch["attention_mask"].to(DEVICE)
                labels    = batch["labels"].to(DEVICE)

                logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
                preds  = logits.argmax(dim=-1)

                mask = labels != -100
                all_preds.extend(preds[mask].cpu().numpy())
                all_labels.extend(labels[mask].cpu().numpy())

        macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        print(f"    Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}  macro-F1={macro_f1:.4f}")

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"    Early stop at epoch {epoch+1}")
                break

    # Cleanup
    del model, optimizer, scheduler, train_loader, val_loader
    torch.cuda.empty_cache()

    return best_f1


# ── Optuna objective ───────────────────────────────────────────
def objective(trial):
    config = {
        "lr":            trial.suggest_categorical("lr", [2e-5, 3e-5, 5e-5, 7e-5]),
        "batch_size":    trial.suggest_categorical("batch_size", [8, 16]),
        "warmup_ratio":  trial.suggest_float("warmup_ratio", 0.0, 0.10),
        "freeze_layers": trial.suggest_categorical("freeze_layers", [0, 2, 4, 6, 8]),
    }
    print(f"\n{'='*60}")
    print(f"Trial {trial.number} | {config}")
    print(f"{'='*60}")

    t0 = time.time()
    f1 = train_and_evaluate(config, train_dataset, val_dataset, CLASS_WEIGHTS)
    elapsed = (time.time() - t0) / 3600

    # Backup log to JSON on Drive
    log_trial(trial, f1, elapsed)

    print(f"  → macro-F1 = {f1:.4f}  ({elapsed:.2f} h)")
    return f1


def log_trial(trial, f1, hours):
    entry = {
        "trial": trial.number,
        "params": trial.params,
        "f1": round(f1, 5),
        "hours": round(hours, 3),
        "timestamp": datetime.now().isoformat(),
    }
    log_path = Path(TRIAL_LOG)
    try:
        existing = json.loads(log_path.read_text()) if log_path.exists() else []
    except Exception:
        existing = []
    existing.append(entry)
    log_path.write_text(json.dumps(existing, indent=2))

In [ ]:
# ── Create / Resume study ──────────────────────────────────────
study = optuna.create_study(
    direction="maximize",
    study_name="mbert_armenian_punct",
    storage=STUDY_DB,
    load_if_exists=True)

completed = len([t for t in study.trials
                 if t.state == optuna.trial.TrialState.COMPLETE])
remaining = max(0, N_TRIALS - completed)

print(f"Completed trials: {completed}/{N_TRIALS}")
print(f"Remaining: {remaining}")

if remaining > 0:
    est_hours = remaining * 1.8
    print(f"Estimated time for remaining trials: ~{est_hours:.1f} h")
    print(f"Trials this session (12h limit): ~{min(remaining, 6)}")
    print(f"\nStarting Optuna...")
    study.optimize(objective, n_trials=remaining)
    print(f"\nOptuna complete! {len(study.trials)} total trials.")
else:
    print("All trials already completed! Proceed to Step 4.")

[I 2026-04-02 17:21:01,627] Using an existing study with name 'mbert_armenian_punct' instead of creating a new one.


Completed trials: 14/15
Remaining: 1
Estimated time for remaining trials: ~1.8 h
Trials this session (12h limit): ~1

Starting Optuna...

Trial 20 | {'lr': 5e-05, 'batch_size': 8, 'warmup_ratio': 0.03111597583678993, 'freeze_layers': 6}


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly ini

    Epoch 1/3  loss=0.3472  macro-F1=0.4511
    Epoch 2/3  loss=0.2391  macro-F1=0.4458
    Epoch 3/3  loss=0.1882  macro-F1=0.4694


[I 2026-04-02 18:09:21,157] Trial 20 finished with value: 0.46935663352349977 and parameters: {'lr': 5e-05, 'batch_size': 8, 'warmup_ratio': 0.03111597583678993, 'freeze_layers': 6}. Best is trial 20 with value: 0.46935663352349977.


  → macro-F1 = 0.4694  (0.81 h)

Optuna complete! 21 total trials.


## Step 4: Review Results (run after all trials complete)

In [ ]:
# ── Show all trial results ─────────────────────────────────────
print(f"{'Trial':>5} | {'F1':>7} | {'LR':>7} | {'BS':>3} | {'Warmup':>7} | {'Freeze':>6}")
print("-" * 55)
for t in sorted(study.trials, key=lambda x: x.value or 0, reverse=True):
    if t.state != optuna.trial.TrialState.COMPLETE:
        continue
    p = t.params
    print(f"{t.number:>5} | {t.value:.4f}  | {p['lr']:.0e} | {p['batch_size']:>3} | "
          f"{p['warmup_ratio']:.4f}  | {p['freeze_layers']:>6}")

print(f"\n*** Best trial: #{study.best_trial.number}, "
      f"macro-F1 = {study.best_value:.4f} ***")
print(f"Best params: {study.best_params}")

Trial |      F1 |      LR |  BS |  Warmup | Freeze
-------------------------------------------------------
   20 | 0.4694  | 5e-05 |   8 | 0.0311  |      6
   15 | 0.4689  | 5e-05 |   8 | 0.0017  |      6
   13 | 0.4665  | 5e-05 |   8 | 0.0068  |      6
   14 | 0.4630  | 5e-05 |   8 | 0.0011  |      6
    0 | 0.4615  | 5e-05 |   8 | 0.0189  |      6
   11 | 0.4591  | 7e-05 |   8 | 0.0803  |      8
   18 | 0.4587  | 5e-05 |   8 | 0.0039  |      0
    1 | 0.4578  | 7e-05 |  16 | 0.0260  |      4
    7 | 0.4569  | 3e-05 |  16 | 0.0092  |      2
    6 | 0.4533  | 7e-05 |  16 | 0.0951  |      2
    2 | 0.4499  | 2e-05 |  16 | 0.0464  |      4
    4 | 0.4493  | 3e-05 |   8 | 0.0229  |      8
    9 | 0.4479  | 7e-05 |  16 | 0.0785  |      8
   10 | 0.4368  | 3e-05 |  16 | 0.0395  |      8
    3 | 0.4366  | 3e-05 |  16 | 0.0738  |      8

*** Best trial: #20, macro-F1 = 0.4694 ***
Best params: {'lr': 5e-05, 'batch_size': 8, 'warmup_ratio': 0.03111597583678993, 'freeze_layers': 6}


In [ ]:
# ── JSON backup review ─────────────────────────────────────────
log_path = Path(TRIAL_LOG)
if log_path.exists():
    log_data = json.loads(log_path.read_text())
    print(f"JSON backup has {len(log_data)} trial entries.")
    total_h = sum(e.get("hours", 0) for e in log_data)
    print(f"Total compute time: {total_h:.1f} hours")
else:
    print("No JSON backup found.")

JSON backup has 15 trial entries.
Total compute time: 11.3 hours


## Step 5: Export Best Config for PC

After all trials complete, save the best hyperparameters.  
Copy `mbert_best_config.json` to your PC and use it in `train_hyebert_v3_matched.ipynb`  
with `MODEL_NAME = "bert-base-multilingual-cased"`.

In [16]:
best_config = {**study.best_params}
best_config["best_f1"] = round(study.best_value, 5)
best_config["best_trial"] = study.best_trial.number
best_config["model_name"] = MODEL_NAME
best_config["num_labels"] = NUM_LABELS
best_config["max_length"] = MAX_LENGTH
best_config["negative_keep_ratio"] = NEGATIVE_KEEP_RATIO
best_config["total_trials"] = len(study.trials)

with open(CONFIG_OUT, "w") as f:
    json.dump(best_config, f, indent=2)

print(f"Best config saved to: {CONFIG_OUT}")
print(json.dumps(best_config, indent=2))
print()
print("NEXT STEPS:")
print("1. Download mbert_best_config.json from Google Drive")
print("2. On PC, open train_hyebert_v3_matched.ipynb")
print("3. Set MODEL_NAME = 'bert-base-multilingual-cased'")
print("4. Load best config from the JSON file")
print("5. Adjust batch_size for dual-GPU (bs=8 per GPU = 16 total)")
print("6. Train for 10 epochs, patience=2, full filtered dataset")

NameError: name 'study' is not defined

In [17]:
config = json.loads((BASE_DIR / "mbert_best_config.json").read_text())
print(f"Loaded config: {config}")

Loaded config: {'lr': 5e-05, 'batch_size': 8, 'warmup_ratio': 0.03111597583678993, 'freeze_layers': 6, 'best_f1': 0.46936, 'best_trial': 20, 'model_name': 'bert-base-multilingual-cased', 'num_labels': 4, 'max_length': 128, 'negative_keep_ratio': 0.15, 'total_trials': 21}


In [21]:
# ================================================================
# FINAL TRAINING — resumable across sessions
# ================================================================

FINAL_EPOCHS = 10
FINAL_PATIENCE = 2

config = json.loads((BASE_DIR / "mbert_best_config.json").read_text())
SAVE_PATH = str(BASE_DIR / "mbert_best.pt")
CHECKPOINT_PATH = str(BASE_DIR / "mbert_training_checkpoint.pt")

# --- Resume logic ---
start_epoch = 1
best_f1, best_epoch, wait = 0.0, 0, 0
history = {"train_loss": [], "val_f1": []}

# Fresh model
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS).to(DEVICE)

freeze_n = config["freeze_layers"]
if freeze_n > 0:
    for i in range(freeze_n):
        for param in model.bert.encoder.layer[i].parameters():
            param.requires_grad = False

# Optimizer setup
no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}
param_groups = [
    {"params": [p for n, p in model.named_parameters()
                 if p.requires_grad and not any(nd in n for nd in no_decay)],
     "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters()
                 if p.requires_grad and any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,
                        num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(param_groups, lr=config["lr"])
total_steps = len(train_loader) * FINAL_EPOCHS
warmup_steps = int(total_steps * config["warmup_ratio"])

def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / max(1, warmup_steps)
    return max(0.0, float(total_steps - step) / max(1, total_steps - warmup_steps))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS, ignore_index=-100)

# Resume from checkpoint if it exists
ckpt_path = Path(CHECKPOINT_PATH)
if ckpt_path.exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_f1 = ckpt["best_f1"]
    best_epoch = ckpt["best_epoch"]
    wait = ckpt["wait"]
    history = ckpt["history"]
    print(f"RESUMED from epoch {ckpt['epoch']}, best_f1={best_f1:.4f} (epoch {best_epoch})")
else:
    print("Starting fresh training.")

print(f"\nConfig: {config}")
print(f"Epochs: {start_epoch}..{FINAL_EPOCHS}, patience={FINAL_PATIENCE}")
print(f"Train: {len(train_dataset):,}, Val: {len(val_dataset):,}")
print("=" * 60)

for epoch in range(start_epoch, FINAL_EPOCHS + 1):
    t0 = time.time()

    # Train
    model.train()
    epoch_loss, n_batches = 0.0, 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        loss = loss_fn(logits.view(-1, NUM_LABELS), labels.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches

    # Evaluate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
            preds = logits.argmax(dim=-1)
            mask = labels != -100
            all_preds.extend(preds[mask].cpu().numpy())
            all_labels.extend(labels[mask].cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    elapsed = time.time() - t0
    history["train_loss"].append(avg_loss)
    history["val_f1"].append(macro_f1)

    print(f"Epoch {epoch}/{FINAL_EPOCHS} — loss: {avg_loss:.4f}, "
          f"val_F1: {macro_f1:.4f}, time: {elapsed:.0f}s")

    if macro_f1 > best_f1:
        best_f1, best_epoch, wait = macro_f1, epoch, 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> Best model saved (F1={best_f1:.4f})")
    else:
        wait += 1
        if wait >= FINAL_PATIENCE:
            print(f"  Early stop at epoch {epoch}")
            break

    # Save checkpoint to Drive after EVERY epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_f1": best_f1,
        "best_epoch": best_epoch,
        "wait": wait,
        "history": history,
    }, CHECKPOINT_PATH)
    print(f"  Checkpoint saved (epoch {epoch})")

print(f"\nDone! Best: epoch {best_epoch}, macro-F1 = {best_f1:.4f}")
print(f"Weights: {SAVE_PATH}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly ini

Starting fresh training.

Config: {'lr': 5e-05, 'batch_size': 8, 'warmup_ratio': 0.03111597583678993, 'freeze_layers': 6, 'best_f1': 0.46936, 'best_trial': 20, 'model_name': 'bert-base-multilingual-cased', 'num_labels': 4, 'max_length': 128, 'negative_keep_ratio': 0.15, 'total_trials': 21}
Epochs: 1..10, patience=2
Train: 35,517, Val: 11,026
Epoch 1/10 — loss: 0.3735, val_F1: 0.4444, time: 897s
  -> Best model saved (F1=0.4444)
  Checkpoint saved (epoch 1)
Epoch 2/10 — loss: 0.2575, val_F1: 0.4273, time: 897s
  Checkpoint saved (epoch 2)
Epoch 3/10 — loss: 0.2194, val_F1: 0.4479, time: 898s
  -> Best model saved (F1=0.4479)
  Checkpoint saved (epoch 3)
Epoch 4/10 — loss: 0.1884, val_F1: 0.4679, time: 897s
  -> Best model saved (F1=0.4679)
  Checkpoint saved (epoch 4)
Epoch 5/10 — loss: 0.1566, val_F1: 0.4602, time: 897s
  Checkpoint saved (epoch 5)
Epoch 6/10 — loss: 0.1304, val_F1: 0.4719, time: 896s
  -> Best model saved (F1=0.4719)
  Checkpoint saved (epoch 6)
Epoch 7/10 — loss: 0.1

In [4]:
# ================================================================
# FULL EVALUATION (CPU — no GPU needed)
# ================================================================

import json, re, difflib, math, random, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification
from sklearn.metrics import classification_report, f1_score

# --- Config ---
LABEL_MAP = {"O": 0, "COMMA_AFTER": 1, "BUTH_AFTER": 2, "REMOVE_COMMA": 3}
LABEL_LIST = ["O", "COMMA_AFTER", "BUTH_AFTER", "REMOVE_COMMA"]
NUM_LABELS = 4
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128
SEED = 42
DEVICE = torch.device("cpu")

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path("drive/MyDrive/Colab Notebooks/Capstone_Data/armenian_punct")
DATA_DIR = BASE_DIR / "results_clean"
SAVE_PATH = str(BASE_DIR / "mbert_best.pt")

# --- Tokenizer + label pipeline (copy from Step 2) ---
ARMENIAN_PUNCT = '\u055b\u055c\u055d\u055e\u055f\u0589'

def tokenize_armenian(text):
    if not isinstance(text, str) or not text.strip():
        return []
    for ch in ARMENIAN_PUNCT:
        text = text.replace(ch, f' {ch} ')
    return re.findall(r'[\w\u0530-\u058F]+|[^\w\s]', text)

def generate_bilstm_labels(orig_tokens, corr_tokens):
    labels = [0] * len(orig_tokens)
    sm = difflib.SequenceMatcher(None, orig_tokens, corr_tokens)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal': continue
        if tag in ('delete', 'replace'):
            for i in range(i1, i2):
                if orig_tokens[i] == ',':
                    labels[i] = 3
        if tag in ('insert', 'replace'):
            for j in range(j1, j2):
                tok = corr_tokens[j]
                if tok == ',' and i1 > 0:
                    labels[i1 - 1] = 1
                elif tok == '\u055d' and i1 > 0:
                    labels[i1 - 1] = 2
    return labels

def _is_punct(tok):
    if tok in set(ARMENIAN_PUNCT):
        return True
    return not bool(re.search(r'[\w\u0530-\u058F]', tok))

def extract_word_labels(orig_tokens, label_ids):
    words, labels = [], []
    for tok, lbl in zip(orig_tokens, label_ids):
        if _is_punct(tok):
            if lbl == 3 and labels and labels[-1] == 0:
                labels[-1] = 3
            continue
        words.append(tok)
        labels.append(lbl)
    return words, labels

# --- Load val data ---
def load_workers(worker_list):
    records = []
    for wf in worker_list:
        with open(DATA_DIR / wf, "r", encoding="utf-8") as f:
            for line in f:
                records.append(json.loads(line))
    return records

def process_record(rec):
    orig = rec.get("original", "") or ""
    corr = rec.get("corrected", "") or rec.get("corrected_sentence", "") or ""
    if not orig.strip(): return None
    if not corr.strip(): corr = orig
    orig_tokens = tokenize_armenian(orig)
    corr_tokens = tokenize_armenian(corr)
    if not orig_tokens: return None
    raw_labels = generate_bilstm_labels(orig_tokens, corr_tokens)
    words, labels = extract_word_labels(orig_tokens, raw_labels)
    if not words: return None
    return {"words": words, "labels": labels}

raw_val = load_workers(["worker_4_clean.jsonl", "worker_5_clean.jsonl"])
val_data = [r for r in (process_record(rec) for rec in raw_val) if r is not None]
print(f"Val records: {len(val_data)}")

# --- Dataset ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PunctDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=MAX_LENGTH):
        self.samples = []
        for rec in data:
            words, labels = rec["words"], rec["labels"]
            encoding = tokenizer(words, is_split_into_words=True,
                                 max_length=max_length, truncation=True,
                                 padding="max_length", return_tensors="pt")
            word_ids = encoding.word_ids(batch_index=0)
            aligned = []
            prev = None
            for wid in word_ids:
                if wid is None: aligned.append(-100)
                elif wid != prev: aligned.append(labels[wid] if wid < len(labels) else 0)
                else: aligned.append(-100)
                prev = wid
            self.samples.append({
                "input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0),
                "labels": torch.tensor(aligned, dtype=torch.long),
            })
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

val_dataset = PunctDataset(val_data, tokenizer)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
print(f"Val dataset: {len(val_dataset)} samples")

# --- Load model + best weights ---
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS)
model.load_state_dict(torch.load(SAVE_PATH, map_location="cpu", weights_only=True))
model.eval()
print("Model loaded.")

# --- Evaluate ---
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        logits = model(input_ids=batch["input_ids"],
                       attention_mask=batch["attention_mask"]).logits
        preds = logits.argmax(dim=-1)
        mask = batch["labels"] != -100
        all_preds.extend(preds[mask].tolist())
        all_labels.extend(batch["labels"][mask].tolist())

print("=" * 60)
print("mBERT FINAL EVALUATION (4-class, unfiltered val)")
print("=" * 60)
print(classification_report(all_labels, all_preds,
      target_names=LABEL_LIST, digits=3, zero_division=0))

final_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
ins_f1 = f1_score(all_labels, all_preds, labels=[1, 2], average="macro", zero_division=0)
preds_3c = [0 if x == 3 else x for x in all_preds]
labels_3c = [0 if x == 3 else x for x in all_labels]
f1_3c = f1_score(labels_3c, preds_3c, labels=[0, 1, 2], average="macro", zero_division=0)

pc, lc = Counter(all_preds), Counter(all_labels)
print("Distribution:")
for i, name in enumerate(LABEL_LIST):
    print(f"  {name:15s}: {lc.get(i,0):>7,} true, {pc.get(i,0):>7,} pred")

print(f"\nMacro-F1 (4-class): {final_f1:.4f}")
print(f"3-class remap F1:   {f1_3c:.4f}")
print(f"Insertion-only F1:  {ins_f1:.4f}")

print("\n" + "=" * 80)
print("CROSS-MODEL COMPARISON (same label pipeline, unfiltered val)")
print("=" * 80)
print(f"{'Model':<30} {'Cls':>4} {'Macro-F1':>10} {'Ins F1':>8}")
print("-" * 56)
print(f"{'BiLSTM v2':<30} {'4':>4} {'0.5933':>10} {'—':>8}")
print(f"{'HyeBERT v3 (buth-fixed)':<30} {'4':>4} {'0.4110':>10} {'—':>8}")
print(f"{'mBERT (this)':<30} {'4':>4} {final_f1:>10.4f} {ins_f1:>8.4f}")

# Save
results = {
    "model": "mbert_matched_pipeline",
    "best_f1_unfiltered": round(final_f1, 5),
    "f1_3class_remap": round(f1_3c, 5),
    "insertion_f1": round(ins_f1, 5),
    "best_epoch": 7,
    "history": {"val_f1": [0.4444, 0.4273, 0.4479, 0.4679, 0.4602, 0.4719, 0.4854, 0.4753, 0.4791]},
}
with open(str(BASE_DIR / "mbert_final_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {BASE_DIR / 'mbert_final_results.json'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Val records: 11026


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Val dataset: 11026 samples


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly ini

Model loaded.
mBERT FINAL EVALUATION (4-class, unfiltered val)
              precision    recall  f1-score   support

           O      0.997     0.976     0.986    238334
 COMMA_AFTER      0.255     0.662     0.368      1110
  BUTH_AFTER      0.294     0.761     0.424      1951
REMOVE_COMMA      0.128     0.225     0.163       111

    accuracy                          0.972    241506
   macro avg      0.418     0.656     0.485    241506
weighted avg      0.987     0.972     0.978    241506

Distribution:
  O              : 238,334 true, 233,373 pred
  COMMA_AFTER    :   1,110 true,   2,887 pred
  BUTH_AFTER     :   1,951 true,   5,051 pred
  REMOVE_COMMA   :     111 true,     195 pred

Macro-F1 (4-class): 0.4854
3-class remap F1:   0.5928
Insertion-only F1:  0.3960

CROSS-MODEL COMPARISON (same label pipeline, unfiltered val)
Model                           Cls   Macro-F1   Ins F1
--------------------------------------------------------
BiLSTM v2                         4     0.5933 